# Домашние растения: SFT, оценка и отчёт

**Подготовка:** загрузите готовую папку `dist/houseplants` с компьютера в корень **Моего диска**. В ней уже есть код, данные и этот ноутбук. Ожидаемый путь: `Мой диск/houseplants/src/pipeline.py`.

Откройте `colab_pipeline.ipynb` из загруженной папки через Colab, выберите **Runtime → Change runtime type → T4 GPU**, затем **Runtime → Run all**. Подтвердите подключение Google Drive. При другом расположении папки измените только `PROJECT_DIR` в первой ячейке. GitHub и API-ключи не нужны.

Процесс выполнит первый baseline, проверку формата, пробу отдельного судьи, короткое обучение, основное SFT, сравнение до/после и отчёт. Модели скачиваются из Hugging Face, это более 20 GB. Ресурсы Colab зависят от доступности. Результаты и checkpoints сохраняются в Drive. При прерывании повторите **Run all**: завершённые этапы будут пропущены.

Данные синтетические: 288 train (96 ситуаций × 3 формулировки), 12 validation, 36 test. Эксперимент уже выполнен на T4: формат 86,11% → 100%, адаптер сохранён, итоговый отчёт включён в папку report. Содержательные ошибки и ограничения судьи разобраны в отчёте. Этот ноутбук предназначен для воспроизведения.


In [ ]:
# @title Подключить папку проекта
USE_DRIVE = True # @param {type:"boolean"}
PROJECT_DIR = "/content/drive/MyDrive/houseplants" # @param {type:"string"}

from pathlib import Path
import shutil

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

SOURCE_PROJECT = Path(PROJECT_DIR)
required = [
    'src/pipeline.py', 'configs/experiment.json', 'requirements-colab.txt',
    'prompts/system.txt', 'prompts/judge.txt',
    'data/train.jsonl', 'data/validation.jsonl', 'data/test.jsonl',
]
missing = [name for name in required if not (SOURCE_PROJECT / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'В папке {SOURCE_PROJECT} не найдены: {missing}. '
        'Загрузите dist/houseplants в Мой диск или исправьте PROJECT_DIR.'
    )

# Для скорости код и входные данные копируются в локальную среду Colab.
# Результаты обучения сохраняются отдельно в Drive.
PROJECT = Path('/content/houseplants')
PROJECT.mkdir(parents=True, exist_ok=True)
if SOURCE_PROJECT.resolve() != PROJECT.resolve():
    for folder in ['src', 'configs', 'prompts', 'data', 'tests', 'scripts', 'report', 'results']:
        source = SOURCE_PROJECT / folder
        if source.is_dir():
            shutil.copytree(source, PROJECT / folder, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
    for name in ['README.md', 'PLAN.md', 'MODEL_CARD.md',
                 'requirements-colab.txt', 'requirements-dev.txt']:
        source = SOURCE_PROJECT / name
        if source.is_file():
            shutil.copy2(source, PROJECT / name)
print('Проект подготовлен:', PROJECT)


In [ ]:
# @title Сохранение результатов
from pathlib import Path
import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.pipeline import fingerprint
if USE_DRIVE:
    RUN_ROOT = Path('/content/drive/MyDrive/houseplants_runs')
else:
    RUN_ROOT = Path('/content/houseplants_runs')
RUN_DIR = RUN_ROOT / fingerprint()[:12]
RUN_DIR.mkdir(parents=True, exist_ok=True)
fixed_revisions = PROJECT / 'results/experiment/model_revisions.json'
if fixed_revisions.exists():
    import json, shutil
    revision_file = RUN_DIR / 'model_revisions.json'
    if revision_file.exists():
        if json.loads(revision_file.read_text()) != json.loads(fixed_revisions.read_text()):
            raise ValueError('Ревизии в папке результатов отличаются от сохранённого эксперимента.')
    else:
        shutil.copy2(fixed_revisions, revision_file)
print('Результаты и checkpoints:', RUN_DIR)


In [ ]:
# @title Установка фиксированных зависимостей
import subprocess, sys, os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Неиспользуемые расширения старого runtime могут конфликтовать с закреплённым torch.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio', 'torchtext', 'librosa'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(PROJECT / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=False)


In [ ]:
# @title Подготовить файлы моделей: докачка и повтор при обрыве
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '30'
"""Подготовка кэша Hugging Face без изменения модели или параметров эксперимента."""

import json
import time
from pathlib import Path


def is_transient_download_error(error):
    from requests.exceptions import ChunkedEncodingError, ConnectionError, HTTPError, Timeout
    if isinstance(error, (ChunkedEncodingError, ConnectionError, Timeout)):
        return True
    if isinstance(error, HTTPError) and error.response is not None:
        return error.response.status_code == 429 or 500 <= error.response.status_code < 600
    return False


def download_snapshot(model_id, revision, *, download=None, pause=time.sleep, attempts=4):
    if download is None:
        from huggingface_hub import snapshot_download
        download = snapshot_download
    if attempts < 1:
        raise ValueError("attempts must be positive")
    for attempt in range(attempts):
        try:
            return download(
                repo_id=model_id,
                revision=revision,
                allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
                max_workers=1,
            )
        except Exception as error:
            if not is_transient_download_error(error) or attempt + 1 == attempts:
                raise
            seconds = min(5 * 2**attempt, 30)
            print(
                f"Обрыв загрузки {model_id}: {type(error).__name__}. "
                f"Через {seconds} с — попытка {attempt + 2}/{attempts}; "
                "скачанные файлы и незавершённые части сохраняются в кэше.",
                flush=True,
            )
            pause(seconds)


def prefetch_models(run_dir, project):
    """Ревизии берутся из того же файла, который использует pipeline."""
    run_dir, project = Path(run_dir), Path(project)
    from src.modeling import resolve_revisions
    revisions = resolve_revisions(run_dir)
    cfg = json.loads((project / "configs/experiment.json").read_text())
    for model_id in dict.fromkeys([cfg["model_id"], cfg["judge_model_id"]]):
        print("Подготовка файлов модели:", model_id, flush=True)
        download_snapshot(model_id, revisions[model_id])
    print("Файлы моделей в кэше. Pipeline продолжит сохранённые этапы.", flush=True)

prefetch_models(RUN_DIR, PROJECT)


In [ ]:
# @title Полный эксперимент: выполнение и возобновление
import subprocess, sys
try:
    subprocess.run([sys.executable, '-m', 'src.pipeline', '--run-dir', str(RUN_DIR)], cwd=PROJECT, check=True)
finally:
    print('Результаты, включая диагностический архив при ошибке:', RUN_DIR)
    archive = RUN_DIR / 'houseplants_results.zip'
    if archive.exists():
        from IPython.display import FileLink, display
        display(FileLink(str(archive)))


In [ ]:
# @title Скачать результаты
from google.colab import files
archive = RUN_DIR / 'houseplants_results.zip'
if archive.exists():
    files.download(str(archive))
else:
    print('Архива ещё нет. Проверьте ошибку выше; папка:', RUN_DIR)


## После выполнения

Сохраните `houseplants_results.zip` в исходную папку проекта на компьютере и сообщите об этом в чате. По этим реальным результатам можно закончить содержательный разбор и выводы. Архив уже содержит адаптер, данные, метрики, графики, отчёт и логи; большие optimizer-checkpoints остаются в Drive для возобновления.

Если стадия упала: повторите запуск. При систематической ошибке скачайте диагностический архив из папки результатов в Drive и передайте его в папку проекта. Для нового эксперимента с изменённым кодом или настройками используется другой хеш папки, чтобы не смешивать результаты.

Перед сдачей в LMS нужны доступные преподавателю ссылки на репозиторий, данные и веса. Публикация и загрузка в LMS этим ноутбуком не выполняются.
